# OLMo BeeS production segmenter and ranker

This notebook is the full local Ollama production runner for the OLMo BeeS Hugging Face dataset. It processes both train and test, segments and scores both chosen and rejected responses, validates exact source reconstruction, and exports the combined JSONL schema consumed by Method A, B, and C.

The run is deliberately two-pass:

1. Q3 processes every response and checkpoints each item immediately.
2. Only after Q3 has completed the entire selected dataset, Q4 processes the Q3 failures.
3. Fully resolved chosen/rejected pairs are exported atomically.

tqdm displays completed items, valid/failed counts, cache hits, rate, elapsed time, and ETA. Restarting the same run directory resumes from the item ledger and validated result cache without redoing completed inference.


## One-time Ollama setup

The validated primary is the Unsloth dynamic Qwen3.8-27B-UD-Q3_K_XL import tagged locally as qwen3.8:27b-q3. The retained fallback is qwen3.8:27b Q4. Keep only one model loaded because Q4 partially spills to system RAM on this two-GPU workstation.

~~~bash
curl -fsSL https://ollama.com/install.sh | sh
mkdir -p .cache/models/qwen3.8-27b-q3
model_path='.cache/models/qwen3.8-27b-q3/Qwen3.8-27B-UD-Q3_K_XL.gguf'
model_sha='00cf92e666c6af6566996c38c89a44ccdb6449ea25ef0f112a452c853b2a71e2'
if ! echo "$model_sha  $model_path" | sha256sum -c - >/dev/null 2>&1; then
  curl -L --fail --retry 30 --retry-all-errors --retry-delay 5 --continue-at -     --output "$model_path"     'https://huggingface.co/unsloth/Qwen3.8-27B-GGUF/resolve/main/Qwen3.8-27B-UD-Q3_K_XL.gguf?download=true'
fi
echo "$model_sha  $model_path" | sha256sum -c -
ollama create qwen3.8:27b-q3 -f Modelfile.qwen3.8-27b-q3
ollama pull qwen3.8:27b
~~~

Start Ollama with the production memory settings:

~~~bash
OLLAMA_NUM_PARALLEL=1 OLLAMA_MAX_LOADED_MODELS=1 OLLAMA_FLASH_ATTENTION=1 OLLAMA_KV_CACHE_TYPE=q8_0 OLLAMA_CONTEXT_LENGTH=12288 ollama serve
~~~

The API always sends think: false. The 12,288-token context and 2,560-token output reserve are preflight-checked against every selected response before inference begins.


In [1]:
# Set once only if this Python environment lacks the dependencies.
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    import subprocess
    import sys

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "datasets",
            "requests",
            "pandas",
            "tqdm",
        ],
        check=True,
    )


In [2]:
from dataclasses import replace
from pathlib import Path
import json
import os
import sys
import textwrap

import pandas as pd
from IPython.display import display


def locate_workspace() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (
            (candidate / "scripts/ranking/ollama_olmo_bees_ranker.py").is_file()
            and (candidate / "notebooks/kaggle/kaggle_ranking.ipynb").is_file()
        ):
            return candidate
    raise FileNotFoundError("Open this notebook from the VPDPO workspace")


WORKSPACE = locate_workspace()
RANKING_SCRIPTS = WORKSPACE / "scripts/ranking"
if str(RANKING_SCRIPTS) not in sys.path:
    sys.path.insert(0, str(RANKING_SCRIPTS))

from ollama_olmo_bees_ranker import (
    OllamaClient,
    OllamaConfig,
    RankingItem,
    ResultCache,
    build_ranking_prompt,
    canonical_prompt_fingerprints,
    load_kaggle_prompt_templates,
    load_olmo_bees_dataset,
    make_ranking_items,
    run_two_pass_production,
    validate_production_items_fit,
)

print("Workspace:", WORKSPACE)


Workspace: /media/fezan/ASi/VPDPO


## Original Kaggle prompts — embedded and verified

The next cell contains complete copies of POSITIVE_PROMPT and NEGATIVE_PROMPT from kaggle_ranking.ipynb. They are not shortened or paraphrased. Before any dataset or model work, the cell compares both copies byte-for-byte with the canonical notebook, checks their indentation, line endings, and __BATCH_INPUT__ placeholders, and renders one chosen and one rejected audit item through the real prompt builder.

The production builder still reads the canonical literals from kaggle_ranking.ipynb; the embedded copies make the prompts visible here and turn accidental drift into an immediate assertion failure.


In [3]:
POSITIVE_PROMPT = textwrap.dedent(r"""
You are a query-conditioned semantic segmentor, scorer, and ranker.

Your job is NOT to answer the question.
Your job is to take an existing ANSWER, split it into short useful spans, score each span from 0.01 to 1.00, and rank all spans by importance for answering the PROMPT.

Core rules:

* Segment by usefulness, not grammar.
* Split a span when isolating it helps separate core content, support, contrast, mechanism, cause, constraint, exact answer, wrapper text, or removable verbosity.
* Do not split only because there is a comma.
* Preserve original wording as much as possible.
* Do not rewrite the answer except for tiny cleanup needed to make a segment natural.

Scoring rules:

* High score = removing the span would seriously weaken the answer.
* Low score = the span is generic, shared background, wrapper text, subject-only, rhetorical glue, or removable.
* 0.95–1.00 = exact answer nucleus.
* 0.80–0.94 = highly important core fact or contrast.
* 0.50–0.79 = useful secondary detail.
* 0.20–0.49 = weak support, shared setup, or abstract framing.
* 0.01–0.19 = wrapper, filler, generic intro, subject-only fragment, or removable wording.

Ranking rules:

* Rank 1 = most important segment.
* Rank N = least important segment.
* Every segment must have one unique rank.
* No ties.
* Ranks must form a complete sequence from 1 to N.
* Rank globally after segmentation.
* Rank by how harmful it would be to remove the segment.
* Scores and ranks must agree: a better rank must always have a higher score.
* Scores must be unique.
* Do not give rank 1 a score of 1.00 unless it truly deserves it.

Comparison rules:

* For difference/comparison prompts, rank by difference axes, not isolated clause wording.
* A difference axis is a dimension like mechanism, structure, purpose, requirement, capability, limitation, treatment, behavior, cause, outcome, or scope.
* Core defining differences outrank practical consequences.
* Practical consequences outrank examples and applications.
* Shared setup ranks low.
* Wrappers and subject-only fragments rank near the bottom.
* Paired contrasts should receive close ranks because they jointly express one difference.
* A dependent clause may outrank its parent only if it contains the actual answer-relevant difference.
* Shared statements like “both are...” should usually score low.
* Clean direct contrast statements should score high.
* Usually keep only the most central 3–5 comparison spans above 0.80.

AXIS SUMMARY RULE:
For comparison prompts, a concise segment that directly names the main difference axis, such as time scale, reliability, persistence, speed, scope, or purpose, should rank highly. It may outrank detailed examples because it states the central distinction directly.

ORDER PRESERVATION RULE:
Return segments in the same order they appear in the original answer. Do not sort by rank or score. The rank column already shows importance. Segment output order must match the answer exactly.

Output only lines in this exact format:
<segment> -> <score> -> <rank>

Do not output headings, bullets, numbering, explanations, labels, JSON, markdown, or extra text.

EXAMPLES

PROMPT:
What is the status of the ozone layer and how it has changed over the years

ANSWER:
The ozone layer, which sits primarily in the stratosphere, plays a critical role in protecting life on Earth by absorbing most of the Sun’s harmful ultraviolet (UV-B) radiation. During the late 20th century, scientists observed significant depletion of this layer, especially over Antarctica, where a seasonal “ozone hole” formed each spring due largely to chlorofluorocarbons (CFCs) and other ozone-depleting substances released by human activity. In response, the international community adopted the Montreal Protocol in 1987, which successfully phased out many of these chemicals. Since then, measurements show that the ozone layer has been gradually recovering, although the process is slow because ozone-depleting compounds remain in the atmosphere for decades. Current projections suggest that the ozone layer could return to near pre-1980 levels around the middle of the 21st century in most regions, with Antarctic recovery expected slightly later. Overall, the ozone layer represents one of the clearest examples of successful global environmental cooperation, even though continued monitoring remains essential.

OUTPUT:
The ozone layer, -> 0.01 -> 25
which sits primarily in the stratosphere, -> 0.02 -> 22
plays a critical role in protecting life on Earth -> 0.04 -> 19
by absorbing most of the Sun’s harmful ultraviolet (UV-B) radiation. -> 0.05 -> 18
During the late 20th century, -> 0.08 -> 15
scientists observed -> 0.03 -> 21
significant depletion of this layer, -> 0.96 -> 2
especially over Antarctica, -> 0.88 -> 4
where a seasonal “ozone hole” formed each spring -> 0.84 -> 5
due largely to chlorofluorocarbons (CFCs) and other ozone-depleting substances released by human activity. -> 0.78 -> 6
In response, -> 0.06 -> 17
the international community adopted the Montreal Protocol in 1987, -> 0.56 -> 9
which successfully phased out many of these chemicals. -> 0.50 -> 10
Since then, -> 0.07 -> 16
measurements show that -> 0.09 -> 14
the ozone layer has been gradually recovering, -> 0.98 -> 1
although the process is slow -> 0.38 -> 11
because ozone-depleting compounds remain in the atmosphere for decades. -> 0.34 -> 12
Current projections suggest that -> 0.10 -> 13
the ozone layer could return to near pre-1980 levels -> 0.74 -> 7
around the middle of the 21st century in most regions, -> 0.62 -> 8
with Antarctic recovery expected slightly later. -> 0.22 -> 23
Overall, -> 0.11 -> 24
the ozone layer represents one of the clearest examples of successful global environmental cooperation, -> 0.12 -> 20
even though continued monitoring remains essential. -> 0.14 -> 3

PROMPT:
Difference between a virus and bacteria

ANSWER:
Viruses and bacteria can both cause illness, but they are very different types of microbes. Bacteria are living single-celled organisms that can grow and reproduce on their own. Viruses are not considered fully living and must infect a host cell to reproduce. Bacterial infections can often be treated with antibiotics, while antibiotics do not work against viruses. Vaccines and antiviral medicines may help prevent or treat some viral infections.

OUTPUT:
Viruses and bacteria -> 0.05 -> 10
can both cause illness, -> 0.12 -> 9
but they are very different types of microbes. -> 0.36 -> 8
Bacteria are living single-celled organisms -> 0.88 -> 4
that can grow and reproduce on their own. -> 0.94 -> 2
Viruses are not considered fully living -> 0.90 -> 3
and must infect a host cell to reproduce. -> 0.96 -> 1
Bacterial infections can often be treated with antibiotics, -> 0.80 -> 6
while antibiotics do not work against viruses. -> 0.84 -> 5
Vaccines and antiviral medicines may help prevent or treat some viral infections. -> 0.58 -> 7

PROMPT:
Difference between RAM and storage

ANSWER:
RAM and storage are both used by a computer to hold data, but they serve different roles. RAM stores data temporarily while programs are running, allowing the CPU to access it quickly. Storage keeps data long term even when the computer is turned off. RAM is much faster but usually smaller in capacity, whereas storage is slower but designed for persistent files, applications, and the operating system.

OUTPUT:
RAM and storage -> 0.06 -> 12
are both used by a computer to hold data, -> 0.14 -> 11
but they serve different roles. -> 0.46 -> 10
RAM stores data temporarily -> 0.94 -> 2
while programs are running, -> 0.78 -> 7
allowing the CPU to access it quickly. -> 0.82 -> 6
Storage keeps data long term -> 0.96 -> 1
even when the computer is turned off. -> 0.90 -> 3
RAM is much faster -> 0.86 -> 4
but usually smaller in capacity, -> 0.62 -> 9
whereas storage is slower -> 0.76 -> 8
but designed for persistent files, applications, and the operating system. -> 0.84 -> 5

PROMPT:
Difference between TCP and UDP

ANSWER:
TCP and UDP are both transport protocols, but they differ in reliability and overhead. TCP is connection-oriented and guarantees ordered delivery. UDP is connectionless and does not guarantee delivery or ordering. TCP has more overhead, while UDP is lighter and better for low-latency use cases.

OUTPUT:
TCP and UDP -> 0.05 -> 9
are both transport protocols, -> 0.10 -> 8
but they differ in reliability and overhead. -> 0.54 -> 7
TCP is connection-oriented -> 0.94 -> 1
and guarantees ordered delivery. -> 0.90 -> 3
UDP is connectionless -> 0.92 -> 2
and does not guarantee delivery or ordering. -> 0.88 -> 4
TCP has more overhead, -> 0.58 -> 6
while UDP is lighter and better for low-latency use cases. -> 0.62 -> 5

PROMPT:
Show about someone telling his children how he met their mother

ANSWER:
The television show you are referring to is How I Met Your Mother. It follows the character Ted Mosby, who narrates the story of his younger years in New York City to his children, explaining the long sequence of events that eventually led him to meet their mother. The narrative structure is distinctive because it is framed as a retrospective story told many years later, allowing the show to shift between timelines and include commentary from the future version of Ted. Over the course of nine seasons, the series explores friendships, relationships, and personal growth through the experiences of Ted and his close group of friends, gradually building toward the reveal of the mother’s identity.

OUTPUT:
The television show you are referring to is -> 0.03 -> 13
How I Met Your Mother. -> 1.00 -> 1
It follows the character Ted Mosby, -> 0.18 -> 6
who narrates the story of his younger years in New York City to his children, -> 0.40 -> 3
explaining the long sequence of events that eventually led him to meet their mother. -> 0.72 -> 2
The narrative structure is distinctive -> 0.06 -> 11
because it is framed as a retrospective story told many years later, -> 0.24 -> 4
allowing the show to shift between timelines -> 0.16 -> 7
and include commentary from the future version of Ted. -> 0.14 -> 8
Over the course of nine seasons, -> 0.04 -> 12
the series explores friendships, relationships, and personal growth -> 0.10 -> 9
through the experiences of Ted and his close group of friends, -> 0.08 -> 10
gradually building toward the reveal of the mother’s identity. -> 0.22 -> 5

PROMPT:
Reverse Fourier transform of [\frac{a - i k}{(a^2 + k^2)\sqrt{2\pi}}]

ANSWER:
To compute the inverse Fourier transform of the function
[
F(k)=\frac{a - i k}{(a^2 + k^2)\sqrt{2\pi}},
]
we can use known transform pairs and linearity properties of the Fourier transform.
A standard result states that
[
\mathcal{F}^{-1}\left[\frac{a}{a^2+k^2}\right]
= \sqrt{\frac{\pi}{2}}\,e^{-a|x|}
]
(up to normalization conventions). Similarly, the inverse transform of
[
\frac{-ik}{a^2+k^2}
]
corresponds to a derivative-related exponential term involving the sign function.
Combining these known identities and accounting for the normalization factor \sqrt{2\pi}, the inverse Fourier transform evaluates to
[
f(x)=e^{-ax}u(x),
]
where u(x) is the unit step function. This result reflects the structure of the transform as a causal exponential signal, which commonly appears in systems analysis and differential-equation solutions.

OUTPUT:
To compute the inverse Fourier transform of the function -> 0.01 -> 19
F(k)=\frac{a - i k}{(a^2 + k^2)\sqrt{2\pi}}, -> 0.68 -> 2
we can use known transform pairs and linearity properties of the Fourier transform. -> 0.04 -> 13
A standard result states that -> 0.02 -> 18
\mathcal{F}^{-1}\left[\frac{a}{a^2+k^2}\right] = \sqrt{\frac{\pi}{2}}\,e^{-a|x|} -> 0.28 -> 4
(up to normalization conventions). -> 0.03 -> 16
Similarly, -> 0.05 -> 17
the inverse transform of -> 0.06 -> 15
\frac{-ik}{a^2+k^2} -> 0.14 -> 6
corresponds to a derivative-related exponential term -> 0.10 -> 8
involving the sign function. -> 0.11 -> 7
Combining these known identities -> 0.07 -> 14
and accounting for the normalization factor \sqrt{2\pi}, -> 0.08 -> 12
the inverse Fourier transform evaluates to -> 0.09 -> 11
f(x)=e^{-ax}u(x), -> 1.00 -> 1
where u(x) is the unit step function. -> 0.18 -> 5
This result reflects the structure of the transform as a causal exponential signal, -> 0.12 -> 10
which commonly appears in systems analysis and differential-equation solutions. -> 0.13 -> 9

PROMPT:
Who produced “All of the Lights” and what accolades he has

ANSWER:
The song “All of the Lights” was produced primarily by Kanye West, who is widely recognized not only as a rapper but also as an influential music producer known for his layered arrangements and genre-blending style. The track appears on his 2010 album My Beautiful Dark Twisted Fantasy and features contributions from several prominent artists.
Kanye West has received numerous accolades throughout his career, including multiple Grammy Awards, recognition for innovation in hip-hop production techniques, and widespread critical acclaim for shaping the sound of modern popular music. In addition to his musical achievements, he has been influential in fashion and cultural commentary, making him one of the most discussed and impactful artists of his generation, although his public career has also included controversy alongside artistic recognition.

OUTPUT:
The song “All of the Lights” was produced primarily by -> 0.06 -> 13
Kanye West, -> 0.98 -> 1
who is widely recognized not only as a rapper -> 0.08 -> 12
but also as an influential music producer -> 0.26 -> 6
known for his layered arrangements and genre-blending style. -> 0.20 -> 8
The track appears on his 2010 album My Beautiful Dark Twisted Fantasy -> 0.18 -> 9
and features contributions from several prominent artists. -> 0.10 -> 10
Kanye West has received numerous accolades throughout his career, -> 0.30 -> 5
including multiple Grammy Awards, -> 0.86 -> 2
recognition for innovation in hip-hop production techniques, -> 0.34 -> 4
and widespread critical acclaim for shaping the sound of modern popular music. -> 0.28 -> 7
In addition to his musical achievements, -> 0.02 -> 15
he has been influential in fashion and cultural commentary, -> 0.04 -> 14
making him one of the most discussed and impactful artists of his generation, -> 0.09 -> 11
although his public career has also included controversy alongside artistic recognition. -> 0.12 -> 3


Return only the final segmented lines grouped under ITEM headers. No explanation.
For every input block, output exactly one matching ITEM header using the same ITEM number as the input.

NOW DO THE SAME TASK.

INPUTS:
__BATCH_INPUT__

""")

NEGATIVE_PROMPT = textwrap.dedent(r"""
You are a query-conditioned semantic segmentor, damage scorer, and ranker.
Your job is NOT to answer the question.
Your job is to take an existing REJECTED / NEGATIVE ANSWER, split it into short useful spans, score each span from 0.01 to 1.00 by how much it damages the quality of the answer, and rank all spans from most damaging to least damaging.
A high score means the span is an important reason the answer is worse.
A low score means the span is correct, harmless, merely stylistically imperfect, generic, or contributes little to why the answer should be rejected.
Core rules
Segment by evaluative effect, not grammar.
Split a span when isolating it helps separate an error, misleading claim, unsupported assertion, contradiction, instruction violation, irrelevant tangent, unnecessary verbosity, correct content, useful support, or harmless framing.
Do not split only because there is a comma.
Preserve the original wording as much as possible.
Do not rewrite or repair the answer except for tiny cleanup needed to make a segment natural.
Judge every segment relative to the PROMPT and the quality of the ANSWER as a whole.
Assume the negative answer is a realistic rejected response: it may contain substantial correct and useful content.
Do NOT treat every sentence in a rejected answer as bad.
Correct or helpful spans should normally receive low damage scores even when they are highly relevant to the prompt.
Central scoring principle
Score according to the following counterfactual:
How much would removing or correcting this span improve the rejected answer?
If removing or correcting the span would substantially improve the answer, give it a high score.
If removing it would barely improve the answer, give it a low score.
If the span is actually correct and useful, it should usually receive a very low score because it is not responsible for the answer being worse.
Do NOT score by importance to answering the question.
Do NOT score by how much information a span contains.
Do NOT score correct central information highly merely because it is important.
The score measures damage, not usefulness.
Scoring rules
0.95–1.00 = answer-defining defect
A central false answer.
A severe misconception.
A direct contradiction of the correct answer.
A major instruction violation that substantially defeats the task.
A fabricated claim that changes the answer.
A wrong conclusion that makes otherwise reasonable reasoning fail.
0.80–0.94 = major damaging flaw
An important factual error.
A misleading central comparison.
Incorrect mechanism or causal explanation.
Strong unsupported claim affecting the conclusion.
Important contradiction.
Major overstatement or incorrect generalization.
0.50–0.79 = meaningful quality reduction
A secondary factual error.
Partially misleading explanation.
Relevant but poorly supported claim.
Noticeable irrelevance.
Overgeneralization that does not destroy the core answer.
Reasoning that points in the wrong direction but is not the final conclusion.
0.20–0.49 = minor defect
Mild imprecision.
Slight overstatement.
Weak support.
Unnecessary tangent.
Noticeable verbosity.
Awkward or somewhat misleading framing that has limited effect on correctness.
0.01–0.19 = little or no damage
Correct information.
Useful explanation.
Harmless setup.
Neutral transition.
Subject-only fragment.
Stylistic wording that does not meaningfully reduce answer quality.
Redundant but harmless information.
Important asymmetry
A correct span can be extremely important to answering the PROMPT and still receive a score near 0.01.
This is intentional.
For negative-response scoring:
importance ≠ damage
For example:
“RAM stores temporary working data”
may be one of the most important facts in an answer about RAM, but if it is correct, it contributes almost nothing to why the answer is rejected and should therefore receive a low damage score.
Conversely:
“RAM keeps its contents permanently when power is removed”
should score very highly because it directly damages the answer.
Damage hierarchy
When deciding which flaws matter most, generally prioritize:
Wrong final answer or conclusion.
False claim that directly answers the prompt.
Incorrect central mechanism, relationship, or comparison.
Contradiction with another important part of the response.
Unsupported or fabricated factual claims.
Important overgeneralization or misleading qualification.
Failure to follow an explicit user instruction.
Relevant but weak or confusing reasoning.
Irrelevant tangents or unnecessary verbosity.
Stylistic imperfections and harmless wording.
This hierarchy is only a guideline. Judge damage in context.
Correct-but-surrounded-by-wrong-content rule
Do not punish a correct span merely because it appears inside a bad answer.
Example:
“Bacteria are living single-celled organisms, while viruses require host cells to reproduce. Antibiotics therefore work against both bacterial and viral infections.”
The first sentence is mostly correct and should score low.
The incorrect claim that antibiotics work against viral infections should score very high.
Separate them.
Partial-truth rule
A span that contains both correct and incorrect information should be split whenever possible.
For example:
“UDP is connectionless and automatically retransmits lost packets.”
Split into:
UDP is connectionless
and
automatically retransmits lost packets.
The first should score low.
The second should score high.
Error-centrality rule
A factual error should not automatically receive a very high score.
Ask whether the error changes the answer materially.
A wrong peripheral date, minor numerical detail, or unnecessary side fact may score only 0.30–0.60.
A false statement that reverses the actual answer may score above 0.90.
Unsupported-certainty rule
Distinguish between:
“X can sometimes lead to Y”
and
“X always causes Y.”
If the evidence only supports a conditional relationship, the unjustified certainty can itself be damaging.
Isolate qualifiers such as:
always
never
definitely
exclusively
proves
guarantees
completely
when they materially create the error.
Contradiction rule
If two spans contradict each other:
Determine which statement is incorrect or misleading relative to the prompt.
Give the incorrect statement the higher damage score.
Do not automatically punish the correct statement simply because another part contradicts it.
If both claims jointly create confusion and neither can independently be identified as correct, both may receive moderately high and nearby scores.
Omission rule
Score only text that actually appears in the ANSWER.
Do NOT invent a segment representing missing information.
An omission can make the overall answer worse, but there may be no span to assign that damage to.
However, if existing wording explicitly causes or reinforces the omission, that wording can be scored.
For example:
“The only difference between TCP and UDP is speed.”
This statement is damaging because it incorrectly excludes other important differences.
Relevance rule
Irrelevant content is damaging only in proportion to how much it interferes with the answer.
A short harmless aside may receive 0.15–0.30.
A long tangent that distracts from or replaces the requested answer may receive 0.50–0.80.
Do not rank harmless verbosity above substantive factual errors.
Instruction-following rule
Explicit user constraints matter.
If the prompt says:
“Answer in one sentence.”
and the answer gives four paragraphs, the extra material damages instruction following.
However, if the factual answer itself is correct, do not label the correct facts as false.
Score the unnecessary expansion according to how strongly it violates the requested format.
Comparison rules
For difference/comparison prompts, judge damage by difference axes.
A difference axis is a dimension such as:
mechanism
structure
purpose
requirement
capability
limitation
persistence
reliability
speed
treatment
behavior
cause
outcome
scope
Incorrect central difference axes should score highly.
Correct differences should score very low even if they are the most important information in the answer.
Incorrect practical consequences usually rank below incorrect defining differences.
Minor examples rank below both.
Shared correct setup should normally score near the bottom.
If two clauses jointly express an incorrect contrast, their scores should usually be close.
If one half of a contrast is correct and the other half is wrong, do not score them equally. The incorrect half should rank much higher.
AXIS ERROR RULE
For comparison prompts, a concise claim that states the wrong main distinction can be the most damaging span in the answer.
For example:
“The main difference between weather and climate is location.”
should score very highly because it replaces the true central distinction with an incorrect one.
FINAL-ANSWER RULE
When a response contains reasoning followed by a clear final conclusion, the final conclusion normally outranks flaws in intermediate reasoning if the conclusion itself is wrong.
Example:
“Therefore, the answer is 12.”
If the correct answer is 8, this may deserve rank 1 even if an earlier arithmetic mistake caused it.
STYLE VS SUBSTANCE RULE
Substantive problems should almost always outrank stylistic ones.
In general:
false answer > misleading reasoning > unsupported assertion > irrelevant content > verbosity > awkward style
Do not assign very high scores merely because wording is inelegant.
REALISTIC NEGATIVE RESPONSE RULE
The ANSWER should be treated like a realistic rejected response from a preference dataset.
It may:
answer most of the question correctly,
contain useful reasoning,
be fluent and coherent,
have only one or two meaningful mistakes,
overstate something,
contain a subtle hallucination,
make an incorrect inference,
include unnecessary information,
imperfectly follow instructions,
or be slightly less precise than a better response.
Do NOT assume rejected answers must be obviously absurd, incoherent, or completely wrong.
A response with one important localized mistake may be a perfectly valid negative DPO example.
Ranking rules
Rank 1 = most damaging segment.
Rank N = least damaging segment.
Every segment must have one unique rank.
No ties.
Ranks must form a complete sequence from 1 to N.
Rank globally after segmentation.
Rank by how much the segment reduces the quality of the response.
Scores and ranks must agree.
A better rank must always have a higher score.
Scores must be unique.
Do not give rank 1 a score of 1.00 unless the span essentially determines why the response is rejected.
Do not artificially force a span above 0.80 if the negative response contains only mild problems.
ORDER PRESERVATION RULE
Return segments in the same order they appear in the original answer.
Do not sort the output by score or rank.
The rank column already expresses damage.
Segment output order must match the ANSWER exactly.
Output format
Output only lines in this exact format:
-> ->
Do not output headings, bullets, numbering, explanations, labels, JSON, markdown, or extra text.

EXAMPLES
EXAMPLE 1
PROMPT:
What is the status of the ozone layer and how has it changed over the years?
ANSWER:
The ozone layer protects Earth from harmful ultraviolet radiation and experienced substantial depletion during the late twentieth century, particularly over Antarctica. The Montreal Protocol greatly reduced emissions of major ozone-depleting substances, and observations since then indicate gradual recovery. The ozone layer has now completely returned to pre-1980 conditions across most of the world, although the Antarctic ozone hole still appears seasonally. Because recovery has already finished in most regions, restrictions on ozone-depleting substances are becoming less important.
OUTPUT:
The ozone layer protects Earth from harmful ultraviolet radiation -> 0.03 -> 12
and experienced substantial depletion during the late twentieth century, -> 0.06 -> 10
particularly over Antarctica. -> 0.04 -> 11
The Montreal Protocol greatly reduced emissions of major ozone-depleting substances, -> 0.02 -> 13
and observations since then indicate gradual recovery. -> 0.01 -> 14
The ozone layer has now completely returned to pre-1980 conditions -> 0.96 -> 1
across most of the world, -> 0.88 -> 2
although the Antarctic ozone hole still appears seasonally. -> 0.08 -> 9
Because recovery has already finished in most regions, -> 0.84 -> 3
restrictions on ozone-depleting substances -> 0.24 -> 7
are becoming less important. -> 0.78 -> 4

EXAMPLE 2
PROMPT:
Difference between a virus and bacteria
ANSWER:
Viruses and bacteria can both cause infectious disease, but they are biologically different. Bacteria are living single-celled organisms that can reproduce independently under suitable conditions. Viruses depend on host cells for replication. Antibiotics are primarily used against bacterial infections, although they can also directly treat some viral infections when the virus has caused severe symptoms. Vaccines and antiviral drugs are instead commonly used for certain viral diseases.
OUTPUT:
Viruses and bacteria -> 0.03 -> 12
can both cause infectious disease, -> 0.04 -> 11
but they are biologically different. -> 0.05 -> 10
Bacteria are living single-celled organisms -> 0.02 -> 13
that can reproduce independently under suitable conditions. -> 0.01 -> 14
Viruses depend on host cells for replication. -> 0.06 -> 9
Antibiotics are primarily used against bacterial infections, -> 0.07 -> 8
although they can also directly treat some viral infections -> 0.96 -> 1
when the virus has caused severe symptoms. -> 0.88 -> 2
Vaccines and antiviral drugs are instead commonly used -> 0.08 -> 7
for certain viral diseases. -> 0.09 -> 6

EXAMPLE 3
PROMPT:
Difference between RAM and storage
ANSWER:
RAM and storage both hold data in a computer, but they have different purposes. RAM is fast working memory used by active programs, while storage keeps files and applications for longer periods. RAM is normally volatile, meaning its contents are lost when power is removed. Storage is non-volatile and retains information without power. Because RAM is faster, increasing RAM always makes a computer faster regardless of what workload is being run.
OUTPUT:
RAM and storage -> 0.02 -> 13
both hold data in a computer, -> 0.04 -> 11
but they have different purposes. -> 0.05 -> 10
RAM is fast working memory -> 0.01 -> 14
used by active programs, -> 0.03 -> 12
while storage keeps files and applications for longer periods. -> 0.06 -> 9
RAM is normally volatile, -> 0.07 -> 8
meaning its contents are lost when power is removed. -> 0.08 -> 7
Storage is non-volatile -> 0.09 -> 6
and retains information without power. -> 0.10 -> 5
Because RAM is faster, -> 0.25 -> 4
increasing RAM always makes a computer faster -> 0.92 -> 1
regardless of what workload is being run. -> 0.86 -> 2

EXAMPLE 4
PROMPT:
Difference between TCP and UDP
ANSWER:
TCP and UDP are both transport-layer protocols. TCP is connection-oriented and provides mechanisms for reliable, ordered delivery. UDP is connectionless and does not itself guarantee that packets will arrive or remain in order. UDP nevertheless automatically retransmits packets whenever loss is detected, just with less overhead than TCP. For that reason it is often preferred for latency-sensitive applications such as real-time voice or gaming.
OUTPUT:
TCP and UDP -> 0.02 -> 13
are both transport-layer protocols. -> 0.03 -> 12
TCP is connection-oriented -> 0.04 -> 11
and provides mechanisms for reliable, ordered delivery. -> 0.01 -> 14
UDP is connectionless -> 0.05 -> 10
and does not itself guarantee that packets will arrive -> 0.06 -> 9
or remain in order. -> 0.07 -> 8
UDP nevertheless automatically retransmits packets -> 0.97 -> 1
whenever loss is detected, -> 0.91 -> 2
just with less overhead than TCP. -> 0.79 -> 3
For that reason -> 0.34 -> 6
it is often preferred for latency-sensitive applications -> 0.08 -> 7
such as real-time voice or gaming. -> 0.09 -> 5

EXAMPLE 5
PROMPT:
Why do seasons occur on Earth?
ANSWER:
Earth experiences seasons mainly because its rotational axis is tilted relative to its orbit around the Sun. During part of the year, one hemisphere is tilted toward the Sun and receives more direct sunlight and longer days, while the opposite hemisphere receives less direct sunlight. Earth's changing distance from the Sun also plays a major role and is responsible for making summers substantially warmer than winters. Together, axial tilt and orbital distance create the four seasons.
OUTPUT:
Earth experiences seasons mainly because its rotational axis is tilted relative to its orbit around the Sun. -> 0.01 -> 12
During part of the year, -> 0.03 -> 10
one hemisphere is tilted toward the Sun -> 0.02 -> 11
and receives more direct sunlight and longer days, -> 0.04 -> 9
while the opposite hemisphere receives less direct sunlight. -> 0.05 -> 8
Earth's changing distance from the Sun -> 0.42 -> 5
also plays a major role -> 0.91 -> 2
and is responsible for making summers substantially warmer than winters. -> 0.97 -> 1
Together, -> 0.14 -> 7
axial tilt and orbital distance -> 0.76 -> 3
create the four seasons. -> 0.58 -> 4

EXAMPLE 6
PROMPT:
What is FastAPI?
ANSWER:
FastAPI is a Python framework for building web APIs. It makes extensive use of Python type hints and can automatically generate interactive API documentation from an application's endpoint definitions. It supports asynchronous request handling and is commonly used for REST-style services. FastAPI itself stores application data in a built-in SQL database, so a separate database such as PostgreSQL or SQLite is generally unnecessary.
OUTPUT:
FastAPI is a Python framework for building web APIs. -> 0.01 -> 12
It makes extensive use of Python type hints -> 0.02 -> 11
and can automatically generate interactive API documentation -> 0.03 -> 10
from an application's endpoint definitions. -> 0.04 -> 9
It supports asynchronous request handling -> 0.05 -> 8
and is commonly used for REST-style services. -> 0.06 -> 7
FastAPI itself stores application data -> 0.94 -> 2
in a built-in SQL database, -> 0.98 -> 1
so a separate database such as PostgreSQL or SQLite -> 0.78 -> 3
is generally unnecessary. -> 0.88 -> 4

EXAMPLE 7
PROMPT:
In one sentence, explain the difference between supervised and unsupervised learning.
ANSWER:
Supervised learning trains on examples paired with target labels, whereas unsupervised learning looks for patterns or structure in data without those target labels. Supervised learning includes tasks such as classification and regression. Unsupervised methods can include clustering and dimensionality reduction. Both are important branches of machine learning and are widely used in industry.
OUTPUT:
Supervised learning trains on examples paired with target labels, -> 0.01 -> 8
whereas unsupervised learning looks for patterns or structure in data without those target labels. -> 0.02 -> 7
Supervised learning includes tasks such as classification and regression. -> 0.31 -> 4
Unsupervised methods can include clustering and dimensionality reduction. -> 0.34 -> 3
Both are important branches of machine learning -> 0.38 -> 2
and are widely used in industry. -> 0.42 -> 1

EXAMPLE 8
PROMPT:
What is overfitting in machine learning and how can it be reduced?
ANSWER:
Overfitting occurs when a model fits the training data very closely but fails to generalize well to unseen data. It can happen when a model is overly complex relative to the amount or diversity of training data. Common approaches to reduce it include regularization, collecting more representative data, data augmentation, early stopping, or reducing model complexity. Increasing model size is another reliable way to reduce overfitting because larger models generalize better.
OUTPUT:
Overfitting occurs when a model fits the training data very closely -> 0.02 -> 12
but fails to generalize well to unseen data. -> 0.01 -> 13
It can happen when a model is overly complex -> 0.03 -> 11
relative to the amount or diversity of training data. -> 0.04 -> 10
Common approaches to reduce it include regularization, -> 0.05 -> 9
collecting more representative data, -> 0.06 -> 8
data augmentation, -> 0.07 -> 7
early stopping, -> 0.08 -> 6
or reducing model complexity. -> 0.09 -> 5
Increasing model size -> 0.48 -> 4
is another reliable way to reduce overfitting -> 0.94 -> 1
because larger models generalize better. -> 0.87 -> 2

NOW DO THE SAME TASK.

INPUTS:
__BATCH_INPUT__
""")

CANONICAL_PROMPTS = load_kaggle_prompt_templates(WORKSPACE / "notebooks/kaggle/kaggle_ranking.ipynb")
assert POSITIVE_PROMPT == CANONICAL_PROMPTS["chosen"], "Positive prompt copy drifted"
assert NEGATIVE_PROMPT == CANONICAL_PROMPTS["rejected"], "Negative prompt copy drifted"

for name, prompt in {
    "POSITIVE_PROMPT": POSITIVE_PROMPT,
    "NEGATIVE_PROMPT": NEGATIVE_PROMPT,
}.items():
    assert "\r" not in prompt, f"{name} contains CR line endings"
    assert "\t" not in prompt, f"{name} contains tab indentation"
    assert textwrap.dedent(prompt) == prompt, f"{name} is not fully dedented"
    assert prompt.count("__BATCH_INPUT__") == 1, f"{name} placeholder count is wrong"
    assert prompt.startswith("\n"), f"{name} lost its canonical leading newline"
    assert prompt.endswith("\n"), f"{name} lost its canonical trailing newline"

audit_items = [
    RankingItem("audit:0:chosen", 0, "chosen", "Audit prompt", "Audit response."),
    RankingItem("audit:0:rejected", 0, "rejected", "Audit prompt", "Audit response."),
]
for item in audit_items:
    rendered = build_ranking_prompt([item])
    canonical_rendered = CANONICAL_PROMPTS[item.side].replace(
        "__BATCH_INPUT__",
        f"\nITEM 0\n\nUSER PROMPT:\n\n{item.prompt}\n\n"
        f"MODEL RESPONSE:\n\n{item.response}\n",
    )
    assert rendered.startswith(canonical_rendered)
    assert "__BATCH_INPUT__" not in rendered
    assert rendered.count("ITEM 0") >= 1

PROMPT_AUDIT = canonical_prompt_fingerprints()
display(pd.DataFrame(PROMPT_AUDIT).T)
print("Both embedded prompt copies exactly match kaggle_ranking.ipynb and render correctly.")


,characters,sha256,batch_placeholders
chosen,14631,2e32f7d12509cca358f15fcb64b0e735c59c59cfac1cdb...,1
rejected,20460,06da3dbfee5bb4f28a435a23c9f49217b97b3aecf294e9...,1


Both embedded prompt copies exactly match kaggle_ranking.ipynb and render correctly.


## Full-run configuration

The defaults cover all 6,000 train rows and 1,891 test rows: 7,891 preference pairs and 15,782 response items. Change RUN_FULL_DATASET to True only after the preflight cell passes. Do not change OUTPUT_DIR when resuming the same run. If the dataset, prompts, models, or generation settings change, use a new output directory; the manifest intentionally refuses to mix incompatible checkpoints.


In [4]:
HF_DATASET_ID = os.environ.get("OLMO_BEES_DATASET_ID", "").strip() or None
LOCAL_DATASET_PATH = WORKSPACE / "artifacts/olmo2_bees/ultrafeedback_bees_olmo2_1b"
DATASET_SPLITS = ("train", "test")

PRIMARY_MODEL = os.environ.get("OLLAMA_PRIMARY_RANKER", "qwen3.8:27b-q3")
FALLBACK_MODEL = os.environ.get("OLLAMA_FALLBACK_RANKER", "qwen3.8:27b")
NUM_CTX = 12288
NUM_PREDICT = 2560
MAX_HTTP_ATTEMPTS = 3
MAX_VALIDATION_ATTEMPTS = 3

PRIMARY_STRATEGY = "single_concurrent"
FALLBACK_STRATEGY = "single_sequential"
CONCURRENT_WORKERS = 2
CHUNK_SIZE = 8
BATCH_SIZE = 2

OUTPUT_DIR = WORKSPACE / "artifacts/ollama_olmo_bees_full"
CACHE_DIR = WORKSPACE / ".cache/ollama_segment_ranker"
RUN_FULL_DATASET = True
REQUIRE_COMPLETE = True
RETRY_FAILED_FALLBACK_ON_RESUME = True

primary_config = OllamaConfig(
    model=PRIMARY_MODEL,
    num_ctx=NUM_CTX,
    num_predict=NUM_PREDICT,
    temperature=0.0,
    top_p=1.0,
    seed=42,
    keep_alive="30m",
    timeout_seconds=600,
    http_max_attempts=MAX_HTTP_ATTEMPTS,
    validation_max_attempts=MAX_VALIDATION_ATTEMPTS,
)
fallback_config = replace(primary_config, model=FALLBACK_MODEL)
primary_client = OllamaClient(primary_config)
fallback_client = OllamaClient(fallback_config)
result_cache = ResultCache(CACHE_DIR)

display(pd.DataFrame([
    {
        "pass": "primary",
        "model": PRIMARY_MODEL,
        "strategy": PRIMARY_STRATEGY,
        "num_ctx": NUM_CTX,
        "num_predict": NUM_PREDICT,
        "think": False,
    },
    {
        "pass": "fallback failures only",
        "model": FALLBACK_MODEL,
        "strategy": FALLBACK_STRATEGY,
        "num_ctx": NUM_CTX,
        "num_predict": NUM_PREDICT,
        "think": False,
    },
]))
print("Output directory:", OUTPUT_DIR)


,pass,model,strategy,num_ctx,num_predict,think
0,primary,qwen3.8:27b-q3,single_concurrent,12288,2560,False
1,fallback failures only,qwen3.8:27b,single_sequential,12288,2560,False


Output directory: /media/fezan/ASi/VPDPO/artifacts/ollama_olmo_bees_full


## Mandatory preflight

This loads every selected split, confirms both Ollama model tags, materializes both response sides, and applies the conservative input/output context guard to every item. It does not run generation.


In [5]:
datasets_by_split = {}
dataset_sources = {}

for split in DATASET_SPLITS:
    dataset, source = load_olmo_bees_dataset(
        split=split,
        dataset_id=HF_DATASET_ID,
        local_path=LOCAL_DATASET_PATH,
    )
    datasets_by_split[split] = dataset
    dataset_sources[split] = source

print("Ollama version:", primary_client.version())
primary_model_info = primary_client.ensure_model()
fallback_model_info = fallback_client.ensure_model()

preflight_items = []
for split, dataset in datasets_by_split.items():
    preflight_items.extend(
        make_ranking_items(
            dataset,
            range(len(dataset)),
            sides=("chosen", "rejected"),
            split=split,
        )
    )

fit_report = validate_production_items_fit(
    preflight_items,
    [primary_config, fallback_config],
)
preflight_summary = {
    "sources": dataset_sources,
    "splits": {split: len(dataset) for split, dataset in datasets_by_split.items()},
    "preference_pairs": sum(len(dataset) for dataset in datasets_by_split.values()),
    "response_items": len(preflight_items),
    "primary_model": primary_model_info.get("name", primary_model_info.get("model")),
    "fallback_model": fallback_model_info.get("name", fallback_model_info.get("model")),
    "prompt_hashes": canonical_prompt_fingerprints(),
    **fit_report,
}
display(pd.Series(preflight_summary, name="preflight"))
assert len(preflight_items) == 2 * sum(len(dataset) for dataset in datasets_by_split.values())
del preflight_items
print("Preflight passed. Set RUN_FULL_DATASET=True in the configuration cell to launch or resume.")


Ollama version: 0.32.14


sources                           {'train': 'local:///media/fezan/ASi/VPDPO/arti...
splits                                                {'train': 6000, 'test': 1891}
preference_pairs                                                               7891
response_items                                                                15782
primary_model                                                        qwen3.8:27b-q3
fallback_model                                                          qwen3.8:27b
prompt_hashes                     {'chosen': {'characters': 14631, 'sha256': '2e...
items                                                                         15782
context_tiers                                                       [(12288, 2560)]
largest_input_item                                              train:5104:rejected
largest_input_tokens_estimate                                                  8917
largest_output_item                                               train:5104

Preflight passed. Set RUN_FULL_DATASET=True in the configuration cell to launch or resume.


## Launch or resume the complete two-pass run

The primary progress bar covers all selected responses. Chosen items are processed together, followed by rejected items, so Ollama can reuse each large canonical prompt prefix; final exports still use source-row order. Valid results are cached atomically and every success or failure is fsynced to pass_events.jsonl before the bar advances. Q4 starts only after the primary bar reaches 100%, and its bar contains only Q3 failures.

Re-running this cell with the same configuration resumes safely. At completion:

- method_abc_train.jsonl and method_abc_test.jsonl contain the combined Method A/B/C schema.
- Point DATA_PATH in the Method A, B, or C training notebook at method_abc_train.jsonl; keep method_abc_test.jsonl for held-out evaluation.
- production_summary.json records counts, models, prompt hashes, and artifact paths.
- unresolved_items.jsonl is empty on a complete run.
- A remaining Q4 failure produces partial JSONL exports and raises instead of silently presenting an incomplete dataset as final. On the next resume, only unresolved Q4 items are tried again; Q3 is not repeated.


In [ ]:
if not RUN_FULL_DATASET:
    print("Full run is armed but not started. Set RUN_FULL_DATASET=True and rerun this cell.")
else:
    production_summary = run_two_pass_production(
        datasets_by_split,
        primary_client,
        fallback_client,
        result_cache,
        OUTPUT_DIR,
        primary_strategy=PRIMARY_STRATEGY,
        fallback_strategy=FALLBACK_STRATEGY,
        chunk_size=CHUNK_SIZE,
        concurrent_workers=CONCURRENT_WORKERS,
        batch_size=BATCH_SIZE,
        show_progress=True,
        require_complete=REQUIRE_COMPLETE,
        retry_failed_fallback_on_resume=RETRY_FAILED_FALLBACK_ON_RESUME,
        unload_when_done=True,
    )
    display(pd.Series(production_summary, name="production"))


primary qwen3.8:27b-q3:   0%|          | 0/15782 [00:00<?, ?item/s]

## Inspect persisted status

This cell is useful after completion or after restarting the kernel. During an active run, the live tqdm bars are the status display.


In [ ]:
summary_path = OUTPUT_DIR / "production_summary.json"
events_path = OUTPUT_DIR / "pass_events.jsonl"

if summary_path.is_file():
    display(pd.Series(json.loads(summary_path.read_text()), name="last summary"))
else:
    print("No production summary yet.")

if events_path.is_file():
    events = pd.read_json(events_path, lines=True)
    latest_events = events.drop_duplicates(["pass", "item_id"], keep="last")
    display(
        latest_events.groupby(["pass", "model", "valid"], dropna=False)
        .size()
        .rename("items")
        .reset_index()
    )
    print("Current item/pass statuses:", len(latest_events))
    print("Durable events including retried items:", len(events))
else:
    print("No pass ledger yet.")
